# Week 77, Day 2: FastAPI

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w077d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('minifast.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of FastAPI (and of Pydantic\'s BaseModel) that runs in the page (Week 77).\n\nNames and behaviour follow FastAPI\'s own, so code written against this module also runs with the\nreal FastAPI (pip install "fastapi[standard]"): FastAPI, BaseModel, Field, Query, Header, Depends,\nHTTPException, JSONResponse, Response and TestClient (from fastapi.testclient on the real thing).\nParameters are read from your type hints: path parameters, query parameters, a JSON body from a\nBaseModel, and dependencies. Bad input gets FastAPI\'s 422 response. It\'s a teaching copy: no\nasync, no OpenAPI page, and only common types (int, float, str, bool, lists, optional, models).\n"""\nimport copy\nimport inspect\nimport json\nimport re\nimport typing\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qsl, urlencode, urlsplit\n\n_MISSING = object()\n\n\n# ---------- validation -------------------------------------------------------------------------\n\nclass ValidationError(Exception):\n    def __init__(self, errors):\n        super().__init__(f"{len(errors)} validation error(s)")\n        self._errors = errors\n\n    def errors(self):\n        return list(self._errors)\n\n\nclass FieldInfo:\n    def __init__(self, default=_MISSING, ge=None, le=None, gt=None, lt=None, min_length=None, max_length=None, alias=None):\n        self.default, self.ge, self.le, self.gt, self.lt = default, ge, le, gt, lt\n        self.min_length, self.max_length, self.alias = min_length, max_length, alias\n\n\ndef Field(default=_MISSING, **kw):\n    return FieldInfo(default, **kw)\n\n\ndef _err(loc, msg, kind):\n    return {"loc": list(loc), "msg": msg, "type": kind}\n\n\ndef _optional(tp):\n    args = typing.get_args(tp)\n    if (typing.get_origin(tp) is typing.Union or type(tp).__name__ == "UnionType") and type(None) in args:\n        rest = [a for a in args if a is not type(None)]\n        return True, rest[0] if len(rest) == 1 else typing.Union[tuple(rest)]\n    return False, tp\n\n\ndef _convert(value, tp, loc, errors, from_text=False):\n    """Convert value to tp, appending errors; returns the value or _MISSING."""\n    optional, tp = _optional(tp)\n    if value is None:\n        if optional:\n            return None\n        errors.append(_err(loc, "Input should not be None", "none_required"))\n        return _MISSING\n    origin = typing.get_origin(tp)\n    if tp is typing.Any or tp is inspect.Parameter.empty:\n        return value\n    if isinstance(tp, type) and issubclass(tp, BaseModel):\n        if isinstance(value, tp):\n            return value\n        if not isinstance(value, dict):\n            errors.append(_err(loc, "Input should be a valid dictionary or object", "model_type"))\n            return _MISSING\n        return tp._validate(value, loc, errors)\n    if origin in (list, tuple, set):\n        if not isinstance(value, (list, tuple)):\n            errors.append(_err(loc, "Input should be a valid list", "list_type"))\n            return _MISSING\n        (item_type,) = typing.get_args(tp)[:1] or (typing.Any,)\n        out = [_convert(v, item_type, list(loc) + [i], errors, from_text) for i, v in enumerate(value)]\n        return _MISSING if any(v is _MISSING for v in out) else origin(out)\n    if tp is bool:\n        if isinstance(value, bool):\n            return value\n        if isinstance(value, str) and value.lower() in ("true", "1", "yes", "on", "false", "0", "no", "off"):\n            return value.lower() in ("true", "1", "yes", "on")\n        errors.append(_err(loc, "Input should be a valid boolean", "bool_parsing"))\n        return _MISSING\n    if tp is int:\n        if isinstance(value, int) and not isinstance(value, bool):\n            return value\n        if isinstance(value, str) and re.fullmatch(r"[+-]?\\d+", value.strip()):\n            return int(value)\n        if isinstance(value, float) and value.is_integer():\n            return int(value)\n        errors.append(_err(loc, "Input should be a valid integer", "int_parsing"))\n        return _MISSING\n    if tp is float:\n        if isinstance(value, (int, float)) and not isinstance(value, bool):\n            return float(value)\n        if isinstance(value, str):\n            try:\n                return float(value)\n            except ValueError:\n                pass\n        errors.append(_err(loc, "Input should be a valid number", "float_parsing"))\n        return _MISSING\n    if tp is str:\n        if isinstance(value, str):\n            return value\n        if from_text:\n            return str(value)\n        errors.append(_err(loc, "Input should be a valid string", "string_type"))\n        return _MISSING\n    if tp is dict or origin is dict:\n        if isinstance(value, dict):\n            return value\n        errors.append(_err(loc, "Input should be a valid dictionary", "dict_type"))\n        return _MISSING\n    return value\n\n\ndef _constraints(value, info, loc, errors):\n    if value is _MISSING or value is None or info is None:\n        return value\n    checks = [("ge", lambda v, n: v >= n, "greater than or equal to", "greater_than_equal"),\n              ("gt", lambda v, n: v > n, "greater than", "greater_than"),\n              ("le", lambda v, n: v <= n, "less than or equal to", "less_than_equal"),\n              ("lt", lambda v, n: v < n, "less than", "less_than")]\n    for attr, ok, words, kind in checks:\n        limit = getattr(info, attr)\n        if limit is not None and not ok(value, limit):\n            errors.append(_err(loc, f"Input should be {words} {limit}", kind))\n            return _MISSING\n    if info.min_length is not None and len(value) < info.min_length:\n        errors.append(_err(loc, f"Value should have at least {info.min_length} item{\'s\' if info.min_length != 1 else \'\'}"\n                           if not isinstance(value, str) else\n                           f"String should have at least {info.min_length} character{\'s\' if info.min_length != 1 else \'\'}",\n                           "string_too_short" if isinstance(value, str) else "too_short"))\n        return _MISSING\n    if info.max_length is not None and len(value) > info.max_length:\n        errors.append(_err(loc, f"String should have at most {info.max_length} characters", "string_too_long"))\n        return _MISSING\n    return value\n\n\nclass BaseModel:\n    """Declare fields as annotations, with defaults or Field(...)."""\n\n    @classmethod\n    def _fields(cls):\n        hints = typing.get_type_hints(cls)\n        fields = {}\n        for name, tp in hints.items():\n            if name.startswith("_"):\n                continue\n            raw = getattr(cls, name, _MISSING)\n            info = raw if isinstance(raw, FieldInfo) else FieldInfo(raw)\n            fields[name] = (tp, info)\n        return fields\n\n    @classmethod\n    def _validate(cls, data, loc, errors):\n        values = {}\n        before = len(errors)\n        for name, (tp, info) in cls._fields().items():\n            key = info.alias or name\n            if key in data:\n                value = _constraints(_convert(data[key], tp, list(loc) + [key], errors), info, list(loc) + [key], errors)\n            elif info.default is not _MISSING:\n                value = copy.deepcopy(info.default)\n            else:\n                errors.append(_err(list(loc) + [key], "Field required", "missing"))\n                value = _MISSING\n            values[name] = value\n        if len(errors) > before:\n            return _MISSING\n        obj = cls.__new__(cls)\n        obj.__dict__.update(values)\n        return obj\n\n    def __init__(self, **data):\n        errors = []\n        obj = type(self)._validate(data, [], errors)\n        if errors:\n            raise ValidationError(errors)\n        self.__dict__.update(obj.__dict__)\n\n    @classmethod\n    def model_validate(cls, data):\n        errors = []\n        obj = cls._validate(data, [], errors)\n        if errors:\n            raise ValidationError(errors)\n        return obj\n\n    def model_dump(self):\n        def dump(v):\n            if isinstance(v, BaseModel):\n                return v.model_dump()\n            if isinstance(v, (list, tuple, set)):\n                return [dump(x) for x in v]\n            return v\n        return {name: dump(getattr(self, name)) for name in type(self)._fields()}\n\n    def __eq__(self, other):\n        return type(other) is type(self) and self.model_dump() == other.model_dump()\n\n    def __repr__(self):\n        inside = " ".join(f"{k}={v!r}" for k, v in self.model_dump().items())\n        return f"{type(self).__name__}({inside})"\n\n\n# ---------- parameters and dependencies ---------------------------------------------------------\n\nclass Query(FieldInfo):\n    pass\n\n\nclass Header(FieldInfo):\n    pass\n\n\nclass _Depends:\n    def __init__(self, dependency):\n        self.dependency = dependency\n\n\ndef Depends(dependency):\n    return _Depends(dependency)\n\n\nclass HTTPException(Exception):\n    def __init__(self, status_code, detail=None, headers=None):\n        super().__init__(detail)\n        self.status_code = status_code\n        self.detail = detail if detail is not None else HTTPStatus(status_code).phrase\n        self.headers = headers or {}\n\n\nclass Response:\n    media_type = "text/plain; charset=utf-8"\n\n    def __init__(self, content=b"", status_code=200, headers=None, media_type=None):\n        self.body = content.encode("utf-8") if isinstance(content, str) else content\n        self.status_code = status_code\n        self.headers = {k.lower(): v for k, v in (headers or {}).items()}\n        if self.body:\n            self.headers.setdefault("content-type", media_type or self.media_type)\n\n\nclass JSONResponse(Response):\n    media_type = "application/json"\n\n    def __init__(self, content=None, status_code=200, headers=None):\n        super().__init__(json.dumps(content).encode("utf-8"), status_code, headers)\n\n\nclass Request:\n    def __init__(self, method, path, query, headers, body):\n        self.method, self.url_path, self.query_params = method, path, dict(query)\n        self.headers = {k.lower(): v for k, v in headers.items()}\n        self.body = body\n\n    def json(self):\n        return json.loads(self.body or b"null")\n\n\n# ---------- the application ---------------------------------------------------------------------\n\nclass FastAPI:\n    def __init__(self, title="FastAPI", version="0.1.0"):\n        self.title, self.version = title, version\n        self.routes = []\n        self.dependency_overrides = {}\n\n    def api_route(self, path, methods, status_code=200, response_model=None):\n        names = re.findall(r"{(\\w+)(?::path)?}", path)\n        regex = re.compile("^" + re.sub(r"{(\\w+)(:path)?}", lambda m: f"(?P<{m.group(1)}>{\'.+\' if m.group(2) else \'[^/]+\'})", path) + "$")\n\n        def decorate(func):\n            for method in methods:\n                self.routes.append((method, regex, names, func, status_code, response_model))\n            return func\n        return decorate\n\n    def get(self, path, **kw):\n        return self.api_route(path, ["GET"], **kw)\n\n    def post(self, path, **kw):\n        return self.api_route(path, ["POST"], **kw)\n\n    def put(self, path, **kw):\n        return self.api_route(path, ["PUT"], **kw)\n\n    def patch(self, path, **kw):\n        return self.api_route(path, ["PATCH"], **kw)\n\n    def delete(self, path, **kw):\n        return self.api_route(path, ["DELETE"], **kw)\n\n    def _solve(self, func, request, path_values, errors, cleanups, cache):\n        func = self.dependency_overrides.get(func, func)\n        if func in cache:\n            return cache[func]\n        hints = typing.get_type_hints(func)\n        kwargs = {}\n        for name, param in inspect.signature(func).parameters.items():\n            tp = hints.get(name, str if param.annotation is inspect.Parameter.empty else param.annotation)\n            default = param.default\n            if isinstance(default, _Depends):\n                kwargs[name] = self._solve(default.dependency, request, path_values, errors, cleanups, cache)\n            elif tp is Request:\n                kwargs[name] = request\n            elif name in path_values:\n                kwargs[name] = _constraints(_convert(path_values[name], tp, ["path", name], errors, True),\n                                            default if isinstance(default, FieldInfo) else None, ["path", name], errors)\n            elif isinstance(tp, type) and issubclass(tp, BaseModel):\n                try:\n                    data = json.loads(request.body) if request.body else _MISSING\n                except ValueError:\n                    errors.append(_err(["body"], "JSON decode error", "json_invalid"))\n                    continue\n                if data is _MISSING:\n                    if default is not inspect.Parameter.empty:\n                        kwargs[name] = default\n                    else:\n                        errors.append(_err(["body"], "Field required", "missing"))\n                    continue\n                kwargs[name] = _convert(data, tp, ["body"], errors)\n            else:\n                info = default if isinstance(default, FieldInfo) else FieldInfo(default if default is not inspect.Parameter.empty else _MISSING)\n                where = "header" if isinstance(default, Header) else "query"\n                key = info.alias or (name.replace("_", "-") if where == "header" else name)\n                if where == "header":\n                    raw = request.headers.get(key.lower(), _MISSING)\n                elif typing.get_origin(_optional(tp)[1]) is list:\n                    found = [v for k, v in request._pairs if k == key]\n                    raw = found if found else _MISSING\n                else:\n                    raw = request.query_params.get(key, _MISSING)\n                if raw is _MISSING:\n                    if info.default is _MISSING:\n                        errors.append(_err([where, key], "Field required", "missing"))\n                    else:\n                        kwargs[name] = info.default\n                    continue\n                kwargs[name] = _constraints(_convert(raw, tp, [where, key], errors, True), info, [where, key], errors)\n        if errors:\n            return _MISSING\n        result = func(**kwargs)\n        if inspect.isgenerator(result):\n            gen = result\n            result = next(gen)\n            cleanups.append(gen)\n        cache[func] = result\n        return result\n\n    def handle(self, method, path, query_pairs, headers, body):\n        request = Request(method, path, query_pairs, headers, body)\n        request._pairs = list(query_pairs)\n        allowed = []\n        for m, regex, names, func, status_code, response_model in self.routes:\n            match = regex.match(path)\n            if not match:\n                continue\n            if m != method:\n                allowed.append(m)\n                continue\n            errors, cleanups = [], []\n            try:\n                try:\n                    result = self._solve(func, request, match.groupdict(), errors, cleanups, {})\n                    if errors:\n                        return JSONResponse({"detail": errors}, 422)\n                except HTTPException as e:\n                    return JSONResponse({"detail": e.detail}, e.status_code, e.headers)\n                return self._respond(result, status_code, response_model)\n            finally:\n                for gen in reversed(cleanups):\n                    try:\n                        next(gen)\n                    except StopIteration:\n                        pass\n        if allowed:\n            return JSONResponse({"detail": "Method Not Allowed"}, 405, {"allow": ", ".join(sorted(set(allowed)))})\n        return JSONResponse({"detail": "Not Found"}, 404)\n\n    @staticmethod\n    def _respond(result, status_code, response_model):\n        if isinstance(result, Response):\n            return result\n        if response_model is not None:\n            data = result.model_dump() if isinstance(result, BaseModel) else result\n            if isinstance(data, list):\n                result = [{k: d[k] for k in response_model._fields()} if isinstance(d, dict) else d.model_dump() for d in\n                          [x.model_dump() if isinstance(x, BaseModel) else x for x in data]]\n            else:\n                result = {k: data[k] for k in response_model._fields()}\n        if status_code == 204:\n            return Response(b"", 204)\n\n        def plain(v):\n            if isinstance(v, BaseModel):\n                return v.model_dump()\n            if isinstance(v, (list, tuple)):\n                return [plain(x) for x in v]\n            if isinstance(v, dict):\n                return {k: plain(x) for k, x in v.items()}\n            return v\n        return JSONResponse(plain(result), status_code)\n\n\nclass _TestResponse:\n    def __init__(self, response):\n        self.status_code = response.status_code\n        self.headers = response.headers\n        self.content = response.body\n\n    @property\n    def text(self):\n        return self.content.decode("utf-8")\n\n    def json(self):\n        return json.loads(self.content)\n\n\nclass TestClient:\n    def __init__(self, app):\n        self.app = app\n\n    def request(self, method, url, params=None, json=None, headers=None, content=None):\n        parts = urlsplit(url)\n        pairs = parse_qsl(parts.query, keep_blank_values=True)\n        if params:\n            pairs += [(k, str(x)) for k, v in params.items() for x in (v if isinstance(v, list) else [v])]\n        body = b""\n        hdrs = {k.lower(): v for k, v in (headers or {}).items()}\n        if json is not None:\n            import json as _json\n            body = _json.dumps(json).encode("utf-8")\n            hdrs.setdefault("content-type", "application/json")\n        elif content is not None:\n            body = content.encode("utf-8") if isinstance(content, str) else content\n        return _TestResponse(self.app.handle(method.upper(), parts.path or "/", pairs, hdrs, body))\n\n    def get(self, url, **kw):\n        return self.request("GET", url, **kw)\n\n    def post(self, url, **kw):\n        return self.request("POST", url, **kw)\n\n    def put(self, url, **kw):\n        return self.request("PUT", url, **kw)\n\n    def patch(self, url, **kw):\n        return self.request("PATCH", url, **kw)\n\n    def delete(self, url, **kw):\n        return self.request("DELETE", url, **kw)\n')

**Predict before you run:** what will this print?



In [ ]:
from minifast import FastAPI, TestClient

app = FastAPI()

@app.get("/items/{item_id}")
def read_item(item_id: int, q: str | None = None):
    return {"item_id": item_id, "doubled": item_id * 2, "q": q}

client = TestClient(app)
print(client.get("/items/21?q=tea").json())
r = client.get("/items/twenty")
print(r.status_code, r.json())

In [ ]:
from minifast import BaseModel, Field, ValidationError

class BookIn(BaseModel):
    title: str = Field(min_length=1)
    author: str
    year: int = Field(ge=1450, le=2100)
    tags: list[str] = []

book = BookIn.model_validate({"title": "Emma", "author": "Austen", "year": "1815"})
print(book, book.model_dump())
try:
    BookIn.model_validate({"title": "", "year": 3000, "tags": "classic"})
except ValidationError as e:
    for error in e.errors():
        print(error["loc"], "-", error["msg"])

In [ ]:
from minifast import FastAPI, Depends, Header, HTTPException, TestClient

app = FastAPI()
log = []
TOKENS = {"t-ada": "ada"}

def get_store():
    log.append("open")
    yield {"notes": ["buy tea"]}
    log.append("close")

def current_user(x_token: str = Header()):
    if x_token not in TOKENS:
        raise HTTPException(401, "invalid token", headers={"WWW-Authenticate": "Bearer"})
    return TOKENS[x_token]

@app.get("/notes")
def notes(user: str = Depends(current_user), store: dict = Depends(get_store)):
    return {"user": user, "notes": store["notes"]}

client = TestClient(app)
print(client.get("/notes", headers={"X-Token": "t-ada"}).json(), log)
r = client.get("/notes", headers={"X-Token": "wrong"})
print(r.status_code, r.json(), r.headers.get("www-authenticate"))
app.dependency_overrides[current_user] = lambda: "test-user"
print(client.get("/notes").json())

> **Key idea.** FastAPI reads endpoints' type hints: names in the path are path parameters; simple types are query parameters (required without a default); Query and Header add constraints and read headers; a Pydantic model parameter is the JSON body. Conversion and validation happen first, so bad input gets 422 with detail errors giving loc, msg and type. Models declare fields with types, defaults and Field constraints; model_validate collects every error; model_dump converts back; response_model filters output (use separate in and out models). HTTPException(status, detail, headers) sends {"detail": ...}. Depends injects shared work (database sessions, the current user, pagination), chained, once per request, with yield for clean-up, and replaceable in tests with dependency_overrides. Real FastAPI is ASGI, supports async def, and documents itself at /docs.

### Exercise: Path and query parameters

Write an app (a module-level `app`) with `GET /products/{product_id}`, where `product_id` is an `int`, and query parameters `currency` (a `str`, default `"GBP"`), `quantity` (an `int` from 1 to 99 with `Query`, default 1) and `gift` (a `bool`, default `False`). Return `{"product_id": ..., "currency": ..., "quantity": ..., "gift": ..., "total": PRICES[product_id] * quantity}` (rounded to 2 places), raising `HTTPException(404, "no product N")` for an unknown product.

In [ ]:
from minifast import FastAPI, Query, HTTPException

PRICES = {1: 3.5, 2: 6.0, 3: 4.25}
app = FastAPI()

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_params():\n    \"type hints become parsing and validation\"\n    from minifast import TestClient\n    client = TestClient(app)\n    assert client.get(\"/products/2?quantity=3&gift=yes\").json() == {\"product_id\": 2, \"currency\": \"GBP\", \"quantity\": 3, \"gift\": True, \"total\": 18.0}\n    assert client.get(\"/products/3\").json()[\"total\"] == 4.25\n    r = client.get(\"/products/9\")\n    assert r.status_code == 404 and r.json() == {\"detail\": \"no product 9\"}\n    r = client.get(\"/products/x?quantity=100\")\n    assert r.status_code == 422 and [e[\"loc\"] for e in r.json()[\"detail\"]] == [[\"path\", \"product_id\"], [\"query\", \"quantity\"]]\n    assert client.get(\"/products/1?quantity=0\").status_code == 422 and client.get(\"/products/1?gift=maybe\").status_code == 422")

### Exercise: Bodies and output models

Write models `UserIn` (`email`: a `str` of at least 3 characters; `name`: a `str` of 1 to 50 characters; `password`: a `str` of at least 8 characters) and `UserOut` (`id`: `int`, `email`, `name`). Then `POST /users` (status `201`, `response_model=UserOut`) storing the user in `USERS` (a dict by id, ids from 1) with the **password replaced by a fake hash** (`"hashed:" + password[::-1]`, standing in for tomorrow's real hashing) and the email lower-cased; a second user with the same email gets `409` with detail `"email already registered"`. And `GET /users/{user_id}` (`response_model=UserOut`, `404` if missing).

In [ ]:
from minifast import FastAPI, BaseModel, Field, HTTPException

app = FastAPI()
USERS = {}

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_models():\n    \"input and output models\"\n    from minifast import TestClient\n    client = TestClient(app)\n    r = client.post(\"/users\", json={\"email\": \"Ada@Example.com\", \"name\": \"Ada\", \"password\": \"correct horse\"})\n    assert r.status_code == 201 and r.json() == {\"id\": 1, \"email\": \"ada@example.com\", \"name\": \"Ada\"}\n    assert \"password\" not in client.get(\"/users/1\").json() and USERS[1][\"password\"] == \"hashed:\" + \"correct horse\"[::-1]\n    assert client.post(\"/users\", json={\"email\": \"ada@example.com\", \"name\": \"A\", \"password\": \"12345678\"}).status_code == 409\n    r = client.post(\"/users\", json={\"email\": \"b@x\", \"name\": \"\", \"password\": \"short\"})\n    assert r.status_code == 422 and sorted(e[\"loc\"][1] for e in r.json()[\"detail\"]) == [\"name\", \"password\"]\n    assert client.post(\"/users\", json={\"name\": \"C\"}).status_code == 422 and client.get(\"/users/9\").status_code == 404")

### Exercise: PUT and PATCH

Write an app storing tasks in `TASKS` (by id), with a model `Task` (`title`: `str` of at least 1 character; `done`: `bool`, default `False`; `priority`: `int` from 1 to 5, default 3). `PUT /tasks/{task_id}` **replaces** the task with a full `Task` body (creating it if it doesn't exist, with `201`; `200` when replacing; return the task). `PATCH /tasks/{task_id}` changes only the fields given in its JSON body: read it from the `Request` (a parameter annotated `Request`, with `.json()`), merge it into the stored task, and **validate the result** with `Task.model_validate`, answering `422` with the errors' detail if it's invalid and `404` if the task doesn't exist. Return the task as a dict.

In [ ]:
from minifast import FastAPI, BaseModel, Field, HTTPException, Request, JSONResponse, ValidationError

app = FastAPI()
TASKS = {}

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_updates():\n    \"full and partial updates\"\n    from minifast import TestClient\n    client = TestClient(app)\n    r = client.put(\"/tasks/7\", json={\"title\": \"Write report\"})\n    assert r.status_code == 201 and r.json() == {\"title\": \"Write report\", \"done\": False, \"priority\": 3}\n    assert client.put(\"/tasks/7\", json={\"title\": \"Write it\", \"priority\": 5}).status_code == 200\n    assert TASKS[7] == {\"title\": \"Write it\", \"done\": False, \"priority\": 5}\n    assert client.patch(\"/tasks/7\", json={\"done\": True}).json() == {\"title\": \"Write it\", \"done\": True, \"priority\": 5}\n    r = client.patch(\"/tasks/7\", json={\"priority\": 9})\n    assert r.status_code == 422 and r.json()[\"detail\"][0][\"loc\"][-1] == \"priority\" and TASKS[7][\"priority\"] == 5\n    assert client.patch(\"/tasks/8\", json={\"done\": True}).status_code == 404\n    assert client.put(\"/tasks/7\", json={\"title\": \"\"}).status_code == 422")

### Exercise: Dependencies

Write three dependencies and an endpoint. `get_db()` yields the module-level `DB` dict and appends `"opened"` and `"closed"` to `DB["log"]` around the request. `current_user(x_token = Header(), db = Depends(get_db))` returns the user name for the token in `db["tokens"]`, raising `HTTPException(401, "invalid token")` otherwise. `pagination(skip: int = Query(0, ge=0), limit: int = Query(10, ge=1, le=50))` returns `{"skip": ..., "limit": ...}`. `GET /messages` (with all three) returns `{"user": USER, "messages": [...]}`: the user's messages from `db["messages"][user]`, sliced by the pagination.

In [ ]:
from minifast import FastAPI, Depends, Header, Query, HTTPException

app = FastAPI()
DB = {"tokens": {"t1": "ada", "t2": "bob"},
      "messages": {"ada": [f"msg {i}" for i in range(25)], "bob": ["hi"]},
      "log": []}

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_depends():\n    \"chained, once per request, with clean-up\"\n    from minifast import TestClient\n    client = TestClient(app)\n    r = client.get(\"/messages?skip=20&limit=3\", headers={\"X-Token\": \"t1\"})\n    assert r.json() == {\"user\": \"ada\", \"messages\": [\"msg 20\", \"msg 21\", \"msg 22\"]}\n    assert DB[\"log\"] == [\"opened\", \"closed\"], \"get_db should run once per request, even though two things need it\"\n    assert client.get(\"/messages\", headers={\"X-Token\": \"t2\"}).json()[\"messages\"] == [\"hi\"]\n    assert client.get(\"/messages\", headers={\"X-Token\": \"nope\"}).status_code == 401 and client.get(\"/messages\").status_code == 422\n    assert client.get(\"/messages?limit=0\", headers={\"X-Token\": \"t1\"}).status_code == 422\n    app.dependency_overrides[current_user] = lambda: \"bob\"\n    try:\n        assert client.get(\"/messages\").json()[\"user\"] == \"bob\"\n    finally:\n        app.dependency_overrides.clear()")

### Exercise: Validating data outside an endpoint

Models are useful anywhere data comes in. Write a model `Order` (`id`: `int`; `customer`: `str` of at least 1 character; `lines`: a list of `Line` models, each with `sku` (a `str`) and `qty` (an `int` of at least 1); `note`: an optional `str`, default `None`). Then `load_orders(rows)`: validate each dict in `rows`, returning `(orders, problems)`: the valid `Order` objects, and for each invalid row a string `"row I: LOC: MSG"` for its **first** error, where `LOC` is the error's `loc` joined with `.` (list positions included).

In [ ]:
from minifast import BaseModel, Field, ValidationError

___

def load_orders(rows):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_validate():\n    \"nested models\"\n    rows = [\n        {\"id\": \"1\", \"customer\": \"Ada\", \"lines\": [{\"sku\": \"T1\", \"qty\": 2}]},\n        {\"id\": 2, \"customer\": \"\", \"lines\": []},\n        {\"id\": 3, \"customer\": \"Bob\", \"lines\": [{\"sku\": \"C1\", \"qty\": 1}, {\"sku\": \"J1\", \"qty\": 0}]},\n        {\"customer\": \"Cy\", \"lines\": []},\n        {\"id\": 5, \"customer\": \"Di\", \"lines\": [], \"note\": \"leave at door\"},\n    ]\n    orders, problems = load_orders(rows)\n    assert [o.id for o in orders] == [1, 5] and orders[0].lines[0].qty == 2 and orders[1].note == \"leave at door\"\n    assert problems == [\"row 1: customer: String should have at least 1 character\",\n                        \"row 2: lines.1.qty: Input should be greater than or equal to 1\",\n                        \"row 3: id: Field required\"]")

### Bug hunt: A path parameter without a type

`book_id` has no type hint, so it arrives as the **string** `"1"`, the dict's keys are integers, and every lookup fails: this prints `404`. Give it a type, so FastAPI converts it and this prints `200 {'title': 'Emma'}`.

In [ ]:
from minifast import FastAPI, HTTPException, TestClient

app = FastAPI()
BOOKS = {1: {"title": "Emma"}}

@app.get("/books/{book_id}")
def get_book(book_id):
    if book_id not in BOOKS:
        raise HTTPException(404)
    return BOOKS[book_id]

r = TestClient(app).get("/books/1")
print(r.status_code, r.json() if r.status_code == 200 else "")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"type hints convert\"\n    assert output.strip() == \"200 {'title': 'Emma'}\", output")

### Bug hunt: Created, but said OK

The endpoint creates something, but the decorator doesn't say so, so it answers `200`: this prints `200`. Set the success status to `201`, so it prints `201`.

In [ ]:
from minifast import FastAPI, BaseModel, TestClient

app = FastAPI()

class Note(BaseModel):
    text: str

@app.post("/notes")
def add_note(note: Note):
    return {"id": 1, "text": note.text}

print(TestClient(app).post("/notes", json={"text": "hi"}).status_code)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"status_code=201\"\n    expect_output(\"201\")")

### Bug hunt: A password hash in the response

The endpoint returns the stored user, password hash included, so it leaks to every client: this prints `['email', 'password_hash']`. Add a `response_model` with only the email, so it prints `['email']`.

In [ ]:
from minifast import FastAPI, BaseModel, TestClient

app = FastAPI()
USERS = {1: {"email": "ada@example.com", "password_hash": "pbkdf2:sha256:..."}}

class UserOut(BaseModel):
    email: str

@app.get("/users/{user_id}")
def get_user(user_id: int):
    return USERS[user_id]

print(sorted(TestClient(app).get("/users/1").json()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an output model\"\n    expect_output(\"['email']\")")

### Puzzle: A FastAPI endpoint

Arrange the lines of a FastAPI endpoint that creates a book from a validated body. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the status FastAPI sends when a body fails validation
q = ___   # the loc of a missing required query parameter named limit, as a list
r = ___   # True or False: a query parameter with no default is required
s = ___   # what dependency_overrides is for
t = ___   # the header name FastAPI reads for a parameter x_api_key = Header()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 422 and q == [\"query\", \"limit\"] and r is True, \"p, q and r\"\n    assert \"test\" in s.lower() or \"fake\" in s.lower() or \"replac\" in s.lower(), \"s\"\n    assert t.lower() == \"x-api-key\", \"t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a reading-list API

Write `create_app(store)`, where `store` is a dict `{"users": {token: name}, "books": {}}`. Dependencies: `current_user` (from an `Authorization: Bearer TOKEN` header, raising `401` with detail `"not authenticated"` and the header `WWW-Authenticate: Bearer` if it's missing or unknown) and `get_store` (returning `store`). Model `BookIn` (`title`: 1 to 200 characters, `author`: at least 1 character, `rating`: optional `int` from 1 to 5). Endpoints, all requiring a user, with each user seeing **only their own** books: `POST /books` (`201`, the book with an `id`; ids count up across all users from 1); `GET /books` with query `author` (optional, any case) and `min_rating` (optional, 1 to 5), returning the list by id; `GET /books/{book_id}` (`404` with detail `"no such book"` for a missing book **or one belonging to someone else**, so ids don't leak who has what); and `DELETE /books/{book_id}` (`204`, or the same `404`).

In [ ]:
from minifast import FastAPI, BaseModel, Field, Depends, Header, Query, HTTPException

def create_app(store):
    app = FastAPI()
    ___
    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_reading_list():\n    \"authentication, ownership and filtering\"\n    from minifast import TestClient\n    store = {\"users\": {\"t-ada\": \"ada\", \"t-bob\": \"bob\"}, \"books\": {}}\n    client = TestClient(create_app(store))\n    ada, bob = {\"Authorization\": \"Bearer t-ada\"}, {\"Authorization\": \"Bearer t-bob\"}\n    r = client.post(\"/books\", json={\"title\": \"Emma\", \"author\": \"Austen\", \"rating\": 5}, headers=ada)\n    assert r.status_code == 201 and r.json() == {\"id\": 1, \"title\": \"Emma\", \"author\": \"Austen\", \"rating\": 5}\n    client.post(\"/books\", json={\"title\": \"Dune\", \"author\": \"Herbert\"}, headers=bob)\n    client.post(\"/books\", json={\"title\": \"Persuasion\", \"author\": \"austen\", \"rating\": 3}, headers=ada)\n    assert [b[\"id\"] for b in client.get(\"/books\", headers=ada).json()] == [1, 3]\n    assert [b[\"id\"] for b in client.get(\"/books?author=AUSTEN&min_rating=4\", headers=ada).json()] == [1]\n    assert client.get(\"/books/2\", headers=ada).status_code == 404 and client.get(\"/books/2\", headers=bob).json()[\"title\"] == \"Dune\"\n    r = client.get(\"/books\")\n    assert r.status_code == 401 and r.json() == {\"detail\": \"not authenticated\"} and r.headers[\"www-authenticate\"] == \"Bearer\"\n    assert client.get(\"/books\", headers={\"Authorization\": \"Bearer nope\"}).status_code == 401\n    assert client.post(\"/books\", json={\"title\": \"\", \"author\": \"X\", \"rating\": 7}, headers=ada).status_code == 422\n    assert client.delete(\"/books/2\", headers=ada).status_code == 404 and client.delete(\"/books/1\", headers=ada).status_code == 204\n    assert client.get(\"/books/1\", headers=ada).status_code == 404\n    assert client.get(\"/books?min_rating=9\", headers=ada).status_code == 422")

## Warm-up

The function never parsed anything: the type hint `item_id: int` told FastAPI to take that part of the path, convert it to an integer, and refuse the request with a `422` and a precise explanation (where, what and why) when it isn't one. `q: str | None = None` became an optional query parameter. That's the whole idea of FastAPI: **declare** what you expect, and the framework enforces it.

## Parameters from type hints

```python
from fastapi import FastAPI, Query, Header

app = FastAPI()

@app.get("/books/{book_id}")
def get_book(book_id: int, include_reviews: bool = False,
             limit: int = Query(10, ge=1, le=100), x_request_id: str | None = Header(None)):
    ...
```

FastAPI looks at each parameter of the function:

- If its name appears in the path (`{book_id}`), it's a **path parameter**, converted to its type.
- If it's a simple type (`int`, `float`, `str`, `bool`, or a `list` of them), it's a **query parameter**: **required** if it has no default, optional otherwise. `bool` accepts `true`, `1`, `yes` and so on.
- **`Query(default, ge=, le=, gt=, lt=, min_length=, max_length=)`** adds constraints; **`Header(...)`** reads a header instead (with `_` in the name becoming `-`: `x_request_id` reads `X-Request-Id`).
- If its type is a **Pydantic model**, it's the **JSON body** (next section).

Decorators set the method and the success status: `@app.post("/books", status_code=201)`. Return a `dict`, a `list` or a model and FastAPI sends JSON; return a `JSONResponse(content, status_code, headers)` for full control.

On your own computer FastAPI also writes **interactive documentation** from all of this: run the app and open `/docs` to see every endpoint, its parameters and its models, and try requests from the browser. Real FastAPI endpoints can be `async def`; FastAPI is built on **ASGI**, the asynchronous successor to WSGI (Week 67's event loops). The copy here runs ordinary functions only.

## Models

A **Pydantic model** describes structured data with type hints:

```python
from pydantic import BaseModel, Field

class BookIn(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    author: str
    year: int = Field(ge=1450, le=2100)
    tags: list[str] = []
```

- Fields without a default are **required**. `Field(...)` adds constraints like `Query`'s.
- Validating converts where it's safe (`"1815"` becomes `1815` for an `int` field) and collects **every** problem rather than stopping at the first: `BookIn.model_validate(data)` raises a `ValidationError` whose `.errors()` lists them, each with a `loc` (where), a `msg` and a `type`.
- Models nest: a field can be another model, or a list of models.
- `.model_dump()` turns a model back into a dict.

In an endpoint, a model-typed parameter **is the body**: `def create(book: BookIn)` receives a validated `BookIn`, and invalid JSON or invalid fields become a `422` automatically.

**Output models.** `@app.get("/users/{id}", response_model=UserOut)` filters what's returned to just `UserOut`'s fields, whatever the function returns. Separate input and output models are a good habit: the input model has the password, the output model never does.

## Errors

- **`raise HTTPException(status_code=404, detail="no book 7")`** stops the endpoint and sends `{"detail": "no book 7"}` with that status; `headers=` can add headers (such as `WWW-Authenticate` on a `401`).
- **Validation errors** get `422 Unprocessable Content` with `{"detail": [...]}`: a list of errors, each with `loc` (such as `["query", "limit"]`, `["path", "book_id"]` or `["body", "year"]`), `msg` and `type`. Clients can show each message next to the right form field.
- An unknown path gets `404` with `{"detail": "Not Found"}`; a known path with the wrong method, `405`.

Because validation happens before your code, an endpoint's body only ever deals with **valid** data. Business rules that types can't express (does this author exist? is this title taken?) still need `HTTPException`s of your own: `404`, `409 Conflict` and so on.

## Dependencies

Many endpoints need the same things: a database session, the current user, pagination settings. FastAPI's **dependency injection** declares them as parameters with **`Depends`**:

```python
def get_db():
    db = SessionLocal()
    try:
        yield db                     # the endpoint runs here
    finally:
        db.close()                   # then this runs, even after an error

def current_user(x_token: str = Header(), db = Depends(get_db)):
    user = db.user_for_token(x_token)
    if user is None:
        raise HTTPException(401, "invalid token")
    return user

@app.get("/me")
def me(user = Depends(current_user)):
    return {"name": user.name}
```

- A dependency is a function whose own parameters are resolved the same way (path, query, headers, body, other dependencies), so they **chain**.
- A dependency that **yields** works like a context manager (Week 61): code before the `yield` runs first, the value is passed in, and code after it runs when the response is ready.
- Within one request, each dependency runs **once**, however many endpoints or other dependencies ask for it.
- In tests, **`app.dependency_overrides[get_db] = fake_db`** swaps a dependency for a fake (Week 72's fakes), with no change to the endpoints.

## Practice

These use the page's `minifast.py`; on your own computer, import from `fastapi` and `pydantic` (and `fastapi.testclient`) instead.

## On your own computer

Install FastAPI and run an app with its development server:

```bash
pip install "fastapi[standard]"
```

```bash
fastapi dev main.py
```

Open `http://127.0.0.1:8000/docs` and try your endpoints there. Then change a function to `async def` and `await` something inside it (an `httpx.AsyncClient` call, say): FastAPI runs it on its event loop. In production, FastAPI runs under an ASGI server such as **Uvicorn** (`uvicorn main:app --workers 4`).

## Quiz

## Daily challenge

## Before you go

Compare today's reading-list API with yesterday's Flask notes API: list everything FastAPI checked for you that you wrote by hand in Flask. Tomorrow: **REST design**, the conventions that make an API predictable: resources, methods, status codes, pagination, versioning and error formats.